# 06 — Clustering, Stability, and Cross-Wave Alignment

This notebook mirrors `python/06_clustering.py` with richer documentation and inline visualisations.

## What this step does

Applies three complementary clustering algorithms to the 10-D UMAP embeddings, evaluates stability, and aligns cluster labels across sessions.

### Algorithms

| Algorithm | Role | Key parameters |
|-----------|------|----------------|
| **HDBSCAN** | Primary hard clustering; discovers arbitrary shapes, assigns noise label (-1) | `min_cluster_size` ∈ {50, 75, 100}, `min_samples` ∈ {10, 20} — best by silhouette |
| **k-medoids** | Secondary hard clustering; no noise class, k explicitly chosen | k ∈ {3, 4, 5, 6, 7} — evaluated by silhouette, Davies-Bouldin, Calinski-Harabasz |
| **GMM** | Soft (probabilistic) assignments using best k from k-medoids sweep | Returns cluster membership probabilities |

### Stability
Bootstrap ARI (100 iterations, 80% resample) quantifies how reproducible the k-medoids solution is.

### Cross-wave alignment
Cluster centroids (in z-scored feature space) are computed per session and aligned across waves using the Hungarian algorithm on cosine distances, so cluster label "1" refers to the same phenotype in both yr2 and yr6.

**Input:** `data/processed/umap_models/umap10_embedding_{label}.npy`, `data/handoff/features_{yr2,yr6}.csv`  
**Output:** `data/handoff/cluster_assignments.csv`, `data/handoff/gmm_probabilities.csv`, `data/handoff/cluster_centroids.csv`

In [ ]:
import sys
from pathlib import Path

# Allow imports from python/ when running notebook from python/notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import logging
import pickle
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hdbscan import HDBSCAN
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn_extra.cluster import KMedoids

from config import (
    HANDOFF_DIR, MODELS_DIR, LOGS_DIR,
    SESSIONS, SESSION_LABELS,
    COL_ID, COL_SESSION,
    HDBSCAN_MIN_CLUSTER_SIZES, HDBSCAN_MIN_SAMPLES,
    KMEDOIDS_K_RANGE,
    N_BOOTSTRAP, BOOTSTRAP_FRACTION,
    RANDOM_STATE,
)

LOGS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "06_clustering.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

print(f"HDBSCAN sweep: min_cluster_size {HDBSCAN_MIN_CLUSTER_SIZES}, min_samples {HDBSCAN_MIN_SAMPLES}")
print(f"k-medoids sweep: k in {list(KMEDOIDS_K_RANGE)}")
print(f"Bootstrap: {N_BOOTSTRAP} iterations, {BOOTSTRAP_FRACTION:.0%} resample fraction")

## Load embeddings

We load the 10-D UMAP embeddings and participant ID arrays saved by notebook 05, and the feature CSVs for centroid computation.

In [ ]:
embeddings       = {}
ids_by_sess      = {}
features_by_sess = {}

for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    embeddings[sess]       = np.load(MODELS_DIR / f"umap10_embedding_{label}.npy")
    ids_by_sess[sess]      = np.load(MODELS_DIR / f"ids_{label}.npy", allow_pickle=True)
    features_by_sess[sess] = pd.read_csv(HANDOFF_DIR / f"features_{label}.csv")
    log.info(f"  {label}: {embeddings[sess].shape[0]} participants, "
             f"{embeddings[sess].shape[1]} UMAP dims")
    print(f"{label}: embedding shape {embeddings[sess].shape}, "
          f"n_participants = {len(ids_by_sess[sess])}")

## HDBSCAN sweep

We try all combinations of `min_cluster_size` and `min_samples` from the config grids. For each run we report k (number of non-noise clusters), noise fraction, and silhouette score. The best configuration (highest silhouette, noise-excluded) is retained per session.

In [ ]:
log.info("HDBSCAN sweep...")
hdbscan_best = {}
hdbscan_tables = {}

for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    X     = embeddings[sess]
    best  = {"sil": -np.inf, "labels": None, "params": None}
    rows  = []

    for mcs, ms in product(HDBSCAN_MIN_CLUSTER_SIZES, HDBSCAN_MIN_SAMPLES):
        labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(X)
        mask   = labels != -1
        k      = len(set(labels[mask]))
        if k < 2 or mask.sum() < 10:
            rows.append({"mcs": mcs, "ms": ms, "k": k,
                         "noise_frac": f"{1 - mask.mean():.1%}", "silhouette": "—"})
            continue
        sil = silhouette_score(X[mask], labels[mask])
        rows.append({"mcs": mcs, "ms": ms, "k": k,
                     "noise_frac": f"{1 - mask.mean():.1%}",
                     "silhouette": f"{sil:.4f}"})
        log.info(f"  {label} mcs={mcs} ms={ms}: k={k} noise={1 - mask.mean():.1%} sil={sil:.3f}")
        if sil > best["sil"]:
            best = {"sil": sil, "labels": labels, "params": (mcs, ms)}

    hdbscan_best[sess]   = best
    hdbscan_tables[sess] = pd.DataFrame(rows)
    log.info(f"  {label} best: {best['params']}  sil={best['sil']:.3f}")
    print(f"\n{label} — HDBSCAN sweep results:")
    print(hdbscan_tables[sess].to_string(index=False))
    print(f"Best: mcs={best['params'][0]}, ms={best['params'][1]}, sil={best['sil']:.4f}")

## k-medoids sweep

k-medoids is run for k = 3 to 7. We evaluate three metrics:
- **Silhouette** (higher = better separated)
- **Davies-Bouldin** (lower = more compact / separated)
- **Calinski-Harabasz** (higher = denser clusters)

The best k is selected by maximum silhouette score.

In [ ]:
log.info("k-medoids sweep...")
kmedoids_results = {}

for sess in SESSIONS:
    label   = SESSION_LABELS[sess]
    X       = embeddings[sess]
    metrics = []

    for k in KMEDOIDS_K_RANGE:
        labels = KMedoids(n_clusters=k, random_state=RANDOM_STATE).fit_predict(X)
        metrics.append({
            "k":                 k,
            "silhouette":        silhouette_score(X, labels),
            "davies_bouldin":    davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "labels":            labels,
        })
        log.info(f"  {label} k={k}: sil={metrics[-1]['silhouette']:.3f} "
                 f"db={metrics[-1]['davies_bouldin']:.3f}")

    best_idx               = int(np.argmax([m["silhouette"] for m in metrics]))
    kmedoids_results[sess] = {"metrics": metrics, "best": metrics[best_idx]}
    log.info(f"  {label} best k={kmedoids_results[sess]['best']['k']}")
    print(f"{label} best k = {kmedoids_results[sess]['best']['k']}")

# Plot silhouette and Davies-Bouldin vs k
n_sess = len(SESSIONS)
fig, axes = plt.subplots(2, n_sess, figsize=(6 * n_sess, 8))

for col_i, sess in enumerate(SESSIONS):
    label   = SESSION_LABELS[sess]
    metrics = kmedoids_results[sess]["metrics"]
    ks      = [m["k"] for m in metrics]
    sils    = [m["silhouette"] for m in metrics]
    dbs     = [m["davies_bouldin"] for m in metrics]
    best_k  = kmedoids_results[sess]["best"]["k"]

    ax_sil = axes[0][col_i] if n_sess > 1 else axes[0]
    ax_db  = axes[1][col_i] if n_sess > 1 else axes[1]

    ax_sil.plot(ks, sils, marker="o", color="steelblue", linewidth=1.5)
    ax_sil.axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
    ax_sil.set_title(f"Silhouette vs k — {label}")
    ax_sil.set_xlabel("k")
    ax_sil.set_ylabel("Silhouette score")
    ax_sil.legend()

    ax_db.plot(ks, dbs, marker="o", color="darkorange", linewidth=1.5)
    ax_db.axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
    ax_db.set_title(f"Davies-Bouldin vs k — {label}")
    ax_db.set_xlabel("k")
    ax_db.set_ylabel("Davies-Bouldin score")
    ax_db.legend()

plt.tight_layout()
plt.show()

## GMM (soft assignments)

A Gaussian Mixture Model is fitted using the best k from the k-medoids sweep. GMM provides **soft (probabilistic) cluster assignments** — each participant receives a probability vector across the k clusters. Hard assignments are the argmax of the probability vector.

In [ ]:
log.info("Fitting GMM...")
gmm_rows = {}

for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    X     = embeddings[sess]
    ids   = ids_by_sess[sess]
    k     = kmedoids_results[sess]["best"]["k"]

    gmm   = GaussianMixture(n_components=k, random_state=RANDOM_STATE, n_init=5)
    gmm.fit(X)
    probs = gmm.predict_proba(X)   # (n, k)
    hard  = gmm.predict(X)

    cluster_sizes = pd.Series(hard).value_counts().sort_index()
    log.info(f"  {label} GMM k={k} done")
    print(f"\n{label} GMM (k={k}) cluster sizes:")
    for ci, n_members in cluster_sizes.items():
        pct = n_members / len(hard)
        print(f"  Cluster {ci + 1}: {n_members:,} ({pct:.1%})")

    gmm_rows[sess] = []
    for j, pid in enumerate(ids):
        row = {COL_ID: pid, COL_SESSION: sess, "wave_label": label}
        for ki in range(k):
            row[f"prob_cluster_{ki + 1}"] = float(probs[j, ki])
        gmm_rows[sess].append(row)

## Bootstrap stability

For each session, 100 bootstrap iterations resample 80% of participants without replacement, fit k-medoids (same best k), and compute the Adjusted Rand Index (ARI) against the full-data solution. Mean ± SD ARI > 0.8 indicates a stable, reproducible clustering.

In [ ]:
log.info(f"Bootstrap stability ({N_BOOTSTRAP} iterations)...")

for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    X     = embeddings[sess]
    k     = kmedoids_results[sess]["best"]["k"]
    full  = kmedoids_results[sess]["best"]["labels"]
    n     = X.shape[0]
    rng   = np.random.default_rng(RANDOM_STATE)
    aris  = []

    for b in range(N_BOOTSTRAP):
        idx  = rng.choice(n, size=int(n * BOOTSTRAP_FRACTION), replace=False)
        boot = KMedoids(n_clusters=k, random_state=int(b)).fit_predict(X[idx])
        aris.append(adjusted_rand_score(full[idx], boot))

    mean_ari = np.mean(aris)
    std_ari  = np.std(aris)
    log.info(f"  {label} ARI: {mean_ari:.3f} ± {std_ari:.3f}")
    print(f"{label}: Bootstrap ARI = {mean_ari:.3f} ± {std_ari:.3f}  (n={N_BOOTSTRAP} iters)")

## Cross-wave alignment

Cluster labels are assigned independently per session, so cluster "1" in yr2 may not correspond to cluster "1" in yr6. We align them using the **Hungarian algorithm** on pairwise cosine distances between cluster centroids (computed in z-scored feature space). The reference session is yr2 (first in `SESSIONS`).

In [ ]:
log.info("Cross-wave alignment...")

z_cols_ref     = None
centroid_rows  = []
labels_raw     = {}
labels_aligned = {}

for sess in SESSIONS:
    label  = SESSION_LABELS[sess]
    feat   = features_by_sess[sess]
    ids    = ids_by_sess[sess]
    lbl    = kmedoids_results[sess]["best"]["labels"]
    z_cols = [c for c in feat.columns if c.startswith("z_")]

    if z_cols_ref is None:
        z_cols_ref = z_cols

    labels_raw[sess] = lbl
    X_feat = feat.set_index(COL_ID).reindex(ids)[z_cols].fillna(0).values

    for k_val in np.unique(lbl):
        centroid = X_feat[lbl == k_val].mean(axis=0)
        row = {"cluster": int(k_val), COL_SESSION: sess, "wave_label": label}
        for j, col in enumerate(z_cols):
            row[col] = float(centroid[j])
        centroid_rows.append(row)

# Align to reference session
ref_sess = SESSIONS[0]

def _centroids(feat, ids, labels, z_cols):
    X = feat.set_index(COL_ID).reindex(ids)[z_cols].fillna(0).values
    return np.array([X[labels == k].mean(axis=0) for k in sorted(np.unique(labels))])

ref_c = _centroids(features_by_sess[ref_sess], ids_by_sess[ref_sess],
                    labels_raw[ref_sess], z_cols_ref)
labels_aligned[ref_sess] = labels_raw[ref_sess].copy()

for sess in SESSIONS[1:]:
    tgt_c  = _centroids(features_by_sess[sess], ids_by_sess[sess],
                         labels_raw[sess], z_cols_ref)
    cos    = cdist(ref_c, tgt_c, metric="cosine")
    ri, ti = linear_sum_assignment(cos)
    remap  = {int(old): int(new) for new, old in zip(ri, ti)}
    labels_aligned[sess] = np.array([remap.get(int(l), int(l))
                                      for l in labels_raw[sess]])
    log.info(f"  Alignment {SESSION_LABELS[ref_sess]}→{SESSION_LABELS[sess]}: {remap}")
    print(f"Alignment mapping {SESSION_LABELS[ref_sess]} → {SESSION_LABELS[sess]}: {remap}")

print("Cross-wave alignment complete.")

## Visualise clusters

Scatter plots of the 2-D UMAP projection (from `umap_coords.csv`) coloured by the aligned cluster label per session.

In [ ]:
umap_coords = pd.read_csv(HANDOFF_DIR / "umap_coords.csv")

# Build aligned-labels lookup
assign_lookup = {}
for sess in SESSIONS:
    for pid, lbl in zip(ids_by_sess[sess], labels_aligned[sess]):
        assign_lookup[(pid, sess)] = int(lbl)

umap_coords["cluster_aligned"] = umap_coords.apply(
    lambda r: assign_lookup.get((r[COL_ID], r[COL_SESSION]), -1), axis=1
)

sessions_present = umap_coords[COL_SESSION].unique()
n_sess = len(sessions_present)

# Determine colour palette (one colour per cluster integer)
all_clusters = sorted(umap_coords["cluster_aligned"].unique())
cmap = plt.get_cmap("tab10", len(all_clusters))
color_map = {c: cmap(i) for i, c in enumerate(all_clusters)}

fig, axes = plt.subplots(1, n_sess, figsize=(7 * n_sess, 5))
if n_sess == 1:
    axes = [axes]

for ax, sess in zip(axes, sessions_present):
    label = SESSION_LABELS.get(sess, sess)
    sub   = umap_coords[umap_coords[COL_SESSION] == sess]

    for cluster_id in sorted(sub["cluster_aligned"].unique()):
        pts = sub[sub["cluster_aligned"] == cluster_id]
        c_label = f"Cluster {cluster_id + 1}" if cluster_id >= 0 else "Noise"
        color   = color_map[cluster_id]
        alpha   = 0.25 if cluster_id == -1 else 0.55
        ax.scatter(pts["umap_1"], pts["umap_2"],
                   s=8, alpha=alpha, c=[color], label=c_label, linewidths=0)

    ax.set_title(f"2-D UMAP clusters — {label}  (n={len(sub):,})")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(markerscale=3, fontsize=8, loc="best")

plt.tight_layout()
plt.show()

## Save

In [ ]:
# --- cluster_assignments.csv ---
assign_rows = []
for sess in SESSIONS:
    label = SESSION_LABELS[sess]
    ids   = ids_by_sess[sess]
    for j, pid in enumerate(ids):
        assign_rows.append({
            COL_ID:            pid,
            COL_SESSION:       sess,
            "wave_label":      label,
            "cluster_hard":    int(labels_raw[sess][j]),
            "cluster_aligned": int(labels_aligned[sess][j]),
        })

pd.DataFrame(assign_rows).to_csv(
    HANDOFF_DIR / "cluster_assignments.csv", index=False)
log.info(f"Saved cluster assignments → {HANDOFF_DIR / 'cluster_assignments.csv'}")
print(f"Saved cluster_assignments.csv  ({len(assign_rows):,} rows)")

# --- gmm_probabilities.csv ---
gmm_all = [row for rows in gmm_rows.values() for row in rows]
pd.DataFrame(gmm_all).to_csv(
    HANDOFF_DIR / "gmm_probabilities.csv", index=False)
log.info(f"Saved GMM probabilities → {HANDOFF_DIR / 'gmm_probabilities.csv'}")
print(f"Saved gmm_probabilities.csv    ({len(gmm_all):,} rows)")

# --- cluster_centroids.csv ---
pd.DataFrame(centroid_rows).to_csv(
    HANDOFF_DIR / "cluster_centroids.csv", index=False)
log.info(f"Saved cluster centroids → {HANDOFF_DIR / 'cluster_centroids.csv'}")
print(f"Saved cluster_centroids.csv    ({len(centroid_rows):,} rows)")

log.info("Clustering complete.")